# 동적 웹페이지 크롤링  : (4) Infinite Scroll
- https://webscraper.io/test-sites/e-commerce/scroll/computers/laptops

# 라이브러리 불러오기

In [1]:
from datetime import datetime
from pathlib import Path

import pandas as pd

from selenium import webdriver
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# 기본 설정

In [2]:
## URL
TARGET_URL = 'https://webscraper.io/test-sites/e-commerce/scroll/computers/laptops'

## 동적 요소 최대 대기 시간(초)
WAIT_TIMEOUT = 10

## 브라우저 화면 표시 여부
## False : 브라우저 화면 표시
## True : 브라우저 화면을 표시하지 않고 실행
HEADLESS = False

## csv 저장 폴더
PROJECT_DIR = Path.cwd().resolve().parents[1]
OUTPUT_DIR = PROJECT_DIR / 'data' / 'dynamic'

## 상품 영역 CSS 선택자
PRODUCT_SELECTOR = 'div.product-wrapper'

## 최대 스크롤 횟수
MAX_SCROLL = 20

## 연속으로 상품 증가가 없을 때 종료할 횟수
MAX_NO_GROWTH = 2

# Chrome WebDriver 생성

In [3]:
def create_driver(headless: bool = HEADLESS) -> webdriver.Chrome:
    """
    Chrome WebDriver를 생성하여 반환한다.

    Args:
        headless:
            True이면 브라우저 화면을 표시하지 않고 실행한다.

    Returns:
        Chrome WebDriver 객체    
    """
    options = Options()

    if headless:
        options.add_argument('--headless=new')

    ## 창 최대화 옵션 추가
    options.add_argument('--start-maximized')

    return webdriver.Chrome(options=options)

# Infinite Scroll 페이지 접속

In [4]:
driver = create_driver()

In [5]:
driver.get(TARGET_URL)

In [6]:
wait = WebDriverWait(driver, WAIT_TIMEOUT)

In [7]:
## 초기 상품 요소가 생성될 때까지 대기
wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, PRODUCT_SELECTOR))
)

<selenium.webdriver.remote.webelement.WebElement (session="5e1af8211bb301783f2a34a3b6fdf510", element="f.CE910C931DAAD9ED624EA739E7A97DCA.d.4EA6D62B42855EECFF6FEB16604ED585.e.54")>

In [8]:
print(f'페이지 제목 : {driver.title}')
print(f'현재 URL : {driver.current_url}')

페이지 제목 : Scroll | Web Scraper Test Sites
현재 URL : https://webscraper.io/test-sites/e-commerce/scroll/computers/laptops


# 초기 상품 개수 확인

In [9]:
product_elements = driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR)
print(f'초기 상품 수 : {len(product_elements)}')

초기 상품 수 : 3


# 현재 페이지 높이 확인

## 브라우저 현재 높이 속성

| 속성명 | 의미 | 비고 |
| :--- | :--- | :--- |
| `document.body.scrollHeight` | 문서 전체의 총 높이 | 스크롤 영역 포함 전체 콘텐츠 길이(브라우저 전체 높이 X) |
| `window.innerHeight` | 현재 눈에 보이는 뷰포트 높이 | 가로 스크롤바 제외, 실제 콘텐츠 표시 영역 |
| `window.scrollY` (또는 `pageYOffset`) | 현재 스크롤된 위치 | 맨 위에서부터 얼마나 아래로 내려왔는지 |
| `window.outerHeight` | 브라우저 창 전체 높이 | 주소창, 탭, 테두리 등 포함 |


In [10]:
page_height = driver.execute_script('return document.body.scrollHeight;')
print(f'현재 페이지 높이 : {page_height}px')

현재 페이지 높이 : 1192px


# 페이지 하단으로 한 번 스크롤

## 현재 상품 수와 콘텐츠 높이 저장

In [11]:
before_count = len(driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR))
before_count

3

In [12]:
before_height = driver.execute_script('return document.body.scrollHeight;')
before_height

1192

## 현재 페이지의 가장 아래로 이동

In [13]:
driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')

# 상품 개수 증가 대기

스크롤 직후 바로 상품을 읽지 않는다.

새로운 상품이 로딩되어 상품 개수가 증가할 때까지 기다린다.

In [14]:
try:
    wait.until(
        lambda current_driver: 
        len(current_driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR)) 
        > before_count
    )
except TimeoutException:
    print('대기 시간 동안 상품 수가 증가하지 않았습니다.')

In [17]:
after_count = len(driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR))
after_height = driver.execute_script('return document.body.scrollHeight;')

print(f'스크롤 후 상품 수 : {after_count}')
print(f'스크롤 후 페이지 높이 : {after_height}')
print(f'추가된 상품 수 : {after_count - before_count}')

스크롤 후 상품 수 : 6
스크롤 후 페이지 높이 : 1503
추가된 상품 수 : 3


# 상품 한 건 추출

현재까지 로드된 상품들 중 첫 번째 상품에서 상품명, 가격, 설명, 상세 URL을 추출

In [23]:
product_elements = driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR)
len(product_elements)

first_product = product_elements[0]
first_product

title_element = first_product.find_element(By.CSS_SELECTOR, 'a.title')
title = title_element.get_attribute('title')
detail_url = title_element.get_attribute('href')
price_text = first_product.find_element(By.CSS_SELECTOR, 'h4.price').text
description = first_product.find_element(By.CSS_SELECTOR, 'p.description').text

print(f'상품명: {title}')
print(f'상세 URL: {detail_url}')
print(f'가격: {price_text}')
print(f'설명: {description}')

상품명: Asus VivoBook X441NA-GA190
상세 URL: https://webscraper.io/test-sites/e-commerce/scroll/product/60
가격: $295.99
설명: Asus VivoBook X441NA-GA190 Chocolate Black, 14", Celeron N3450, 4GB, 128GB SSD, Endless OS, ENG kbd


# 상품 요소 한 건을 딕셔너리로 변환

In [24]:
def parse_product_element(product_element: WebElement) -> dict[str: str]:
    """
    상품 WebElement 한 건에서 상품 정보를 추출한다.

    Args:
        product_element:
            상품 영역 WebElement

    Returns:
        상품 정보 딕셔너리
    """
    title_element = product_element.find_element(By.CSS_SELECTOR, 'a.title')
    title = title_element.get_attribute('title')
    detail_url = title_element.get_attribute('href')
    price_text = product_element.find_element(By.CSS_SELECTOR, 'h4.price').text
    description = product_element.find_element(By.CSS_SELECTOR, 'p.description').text

    return {
        'title': title,
        'detail_url' : detail_url,
        'price_text': price_text,
        'description': description,
    }

In [25]:
parse_product_element(first_product)

{'title': 'Asus VivoBook X441NA-GA190',
 'detail_url': 'https://webscraper.io/test-sites/e-commerce/scroll/product/60',
 'price_text': '$295.99',
 'description': 'Asus VivoBook X441NA-GA190 Chocolate Black, 14", Celeron N3450, 4GB, 128GB SSD, Endless OS, ENG kbd'}

# 브라우저 종료

In [26]:
driver.quit()
print('브라우저 종료 🚀')

브라우저 종료 🚀


# [함수 정의] Infinite Scroll 반복 함수

## 종료 조건

Infinite Scroll은 페이지마다 마지막 데이터가 어디인지 명확하지 않을 수 있다.

다음 두 조건 중 하나를 만족하면 반복을 종료한다.
```plaintext
1. MAX_SCROLLS만큼 스크롤한 경우

또는 

2. 연속 MAX_NO_GROWTH회 동안
   상품 개수가 증가하지 않는 경우
```

한 번의 대기 실패만으로 바로 종료하지 않고,
연속으로 증가가 없는지 확인하면 일시적으로 네트워크 지연에도 
조금 더 안정적으로 대응할 수 있다.

In [28]:
def crawl_infinite_scroll_products(
    target_url: str = TARGET_URL,
    wait_timeout: int = WAIT_TIMEOUT,
    max_scrolls: int = MAX_SCROLL,
    max_no_growth: int = MAX_NO_GROWTH,
    headless: bool = HEADLESS,
) -> list[dict[str, str]]:
    """
    Infinite Scroll 페이지에서
    스크롤을 반복하여 전체 상품을 수집한다.

    Args:
        target_url:
            크롤링 대상 URL

        wait_timeout:
            추가 상품 최대 대기 시간(초)

        max_scrolls:
            최대 스크롤 횟수

        max_no_growth:
            상품 증가가 없는 상태를 
            연속으로 허용할 횟수

        headless:
            브라우저 화면 표시 여부

    Returns:
        전체 상품 정보 딕셔너리 정보        
    """

    driver = create_driver(headless=headless)
    wait = WebDriverWait(driver, wait_timeout)

    try:
        driver.get(target_url)

        ## 초기 상품 로딩 대기
        wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, PRODUCT_SELECTOR))
        )

        no_growth_count = 0

        for scroll_count in range(1, MAX_SCROLL + 1):
            ## 스크롤 전 상품 개수
            before_count = len(driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR))

            ## 스크롤 전 페이지 높이
            before_height = driver.execute_script('return document.body.scrollHeight;')

            ## 페이지 가장 아래로 이동
            driver.execute_script(f'window.scrollTo(0, {before_height})')

            try:
                wait.until(
                    lambda current_driver: 
                    len(current_driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR)) 
                    > before_count
                )
            except TimeoutException:
                pass

            ## 스크롤 후 상품 개수
            after_count = len(driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR))

            ## 스크롤 후 페이지 높이
            after_height = driver.execute_script('return document.body.scrollHeight;')

            print(
                f'{scroll_count}회 스크롤 : '
                f'상품 개수 {before_count} -> {after_count}, '
                f'높이 {before_height} -> {after_height}'
            )

            ## 상품이 증가한 경우
            if after_count > before_count:
                no_growth_count = 0
            ## 상품이 증가하지 않은 경우
            else:
                no_growth_count += 1
                print(f'상품 증가 없음 : {no_growth_count}/{max_no_growth}')

            ## 연속으로 일정 횟수 이상 상품 증가가 없으면 반복 종료
            if(no_growth_count >= max_no_growth):
                print('더 이상 추가 상품이 로드되지 않아 종료합니다.')
                break

        ## 최종 화면의 전체 상품 요소 추출
        product_elements = driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR)

        products = [
            parse_product_element(product_element) 
            for product_element 
            in product_elements
        ]

        return products

    finally:
        ## 예외 발생 여부와 관계없이 브라우저 종료
        driver.quit()
    

## Infinite Scroll 크롤링 실행

In [30]:
try:
    products = crawl_infinite_scroll_products()
except TimeoutException as error:
    print(f'오류 내용 : {error}')
    raise

print()
print(f'전체 수집 상품 수 : {len(products)}')

1회 스크롤 : 상품 개수 3 -> 6, 높이 1192 -> 1503
2회 스크롤 : 상품 개수 6 -> 9, 높이 1503 -> 1814
3회 스크롤 : 상품 개수 9 -> 12, 높이 1814 -> 2125
4회 스크롤 : 상품 개수 12 -> 15, 높이 2125 -> 2436
5회 스크롤 : 상품 개수 15 -> 18, 높이 2436 -> 2747
6회 스크롤 : 상품 개수 18 -> 21, 높이 2747 -> 3058
7회 스크롤 : 상품 개수 21 -> 24, 높이 3058 -> 3369
8회 스크롤 : 상품 개수 24 -> 27, 높이 3369 -> 3680
9회 스크롤 : 상품 개수 27 -> 30, 높이 3680 -> 3991
10회 스크롤 : 상품 개수 30 -> 33, 높이 3991 -> 4302
11회 스크롤 : 상품 개수 33 -> 36, 높이 4302 -> 4613
12회 스크롤 : 상품 개수 36 -> 39, 높이 4613 -> 4924
13회 스크롤 : 상품 개수 39 -> 42, 높이 4924 -> 5235
14회 스크롤 : 상품 개수 42 -> 45, 높이 5235 -> 5546
15회 스크롤 : 상품 개수 45 -> 48, 높이 5546 -> 5857
16회 스크롤 : 상품 개수 48 -> 51, 높이 5857 -> 6168
17회 스크롤 : 상품 개수 51 -> 54, 높이 6168 -> 6479
18회 스크롤 : 상품 개수 54 -> 57, 높이 6479 -> 6790
19회 스크롤 : 상품 개수 57 -> 60, 높이 6790 -> 7101
20회 스크롤 : 상품 개수 60 -> 63, 높이 7101 -> 7412

전체 수집 상품 수 : 63


In [31]:
!start .